# Download activations on Colab → save to Google Drive

This notebook **drives** the white-box activation extraction (`deception_detection/scripts/experiment.py`)
from Colab, while **all code stays in GitHub** and **all outputs go to Google Drive**.

```
VS Code (edit) --git push--> GitHub <--git pull-- Colab (run) --> Google Drive (outputs)
```

**How the workflow fits together**
- Edit code in **VS Code** on your laptop, then `git push`.
- In Colab, re-run the *Clone / update repo* cell (`git pull`) to get your changes, then re-run.
- The model runs **remotely on NDIF** (`model.trace(..., remote=True)`), so Colab only orchestrates
  and downloads the activations — you do **not** need a big GPU here.
- Activations + results are written to **Drive**, and the run is **resumable**: activations are cached
  **per batch** with atomic writes, so if Colab disconnects you reconnect and re-run the *Run
  extraction* cell — completed batches/models load from the Drive cache and only the remainder is
  re-downloaded from NDIF.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU.** (Even though compute is remote, a CUDA runtime avoids
   an error in the entry script's memory-logging line. A free T4 is plenty.)
2. Add your keys under **🔑 Secrets** (left sidebar): `NDIF_API_KEY` (from https://login.ndif.us) and
   `HF_TOKEN`. Toggle *Notebook access* on for each.
3. Make sure your target models are **hosted on NDIF** — `remote=True` only works for models NDIF serves.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

Edit these to match your repo, branch, models and config file.

In [ ]:
# --- Repo -------------------------------------------------------------------
REPO_URL = 'https://github.com/zami456/expLB.git'
BRANCH   = 'claude/nnsight-colab-activation-download-f95pop'  # branch you develop on in VS Code
REPO_DIR = '/content/expLB'
# All extraction commands run from the vendored probes package:
WORKDIR  = REPO_DIR + '/src/probes/deception-detection'

# --- Google Drive output locations -----------------------------------------
DRIVE_BASE     = '/content/drive/MyDrive/expLB'
OUT            = DRIVE_BASE + '/results'             # experiment folders (detector.pt, *_scores.json)
ACTS_CACHE_DIR = DRIVE_BASE + '/activations_cache'   # cached NDIF activations -> enables resume

# --- What to run ------------------------------------------------------------
CONFIG_FILE = 'deception_detection/scripts/configs/repe.yaml'
# One run per model (the repo is single-model-per-run). Edit to your models.
#
# IMPORTANT: only Llama and Qwen work with remote NDIF execution -- in models.py they load via
# nnsight LanguageModel, so their weights stay on NDIF. Mistral and Gemma instead load the FULL
# model locally (AutoModelForImageTextToText, device_map='auto', local_files_only=True) and will
# NOT work on Colab. Also confirm the model is actually hosted on NDIF (llama-70b-3.1 is safest).
MODELS = [
    'llama-70b-3.1',   # safest: NDIF's flagship hosted model
    'llama-70b-3.3',
    'qwen-72b',
    # 'mistral-small-3.1-24b',  # loads locally -> NOT supported via remote on Colab
    # 'gemma-3-27b',            # loads locally -> NOT supported via remote on Colab
]

import os
os.makedirs(OUT, exist_ok=True)
os.makedirs(ACTS_CACHE_DIR, exist_ok=True)
print('Outputs   ->', OUT)
print('Acts cache->', ACTS_CACHE_DIR)

## 3. Clone / update the repo

Run this cell every time you push new code from VS Code — it clones on the first run and `git pull`s
afterwards. For a **private** repo, replace `REPO_URL` with a token URL
(`https://<user>:<token>@github.com/zami456/expLB.git`) or use `gh auth`.

In [ ]:
import os, subprocess

if not os.path.isdir(REPO_DIR + '/.git'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

print(subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)

## 4. Install dependencies

Installs the `deception_detection` package from the vendored probes dir — this pulls in `nnsight`
and everything else the extraction needs. Takes a few minutes; you only need to re-run it when
dependencies change.

In [ ]:
!pip install -q -e {WORKDIR}

## 5. Secrets & remote (NDIF) configuration

Reads `NDIF_API_KEY` / `HF_TOKEN` from Colab Secrets, sets them in the environment, points the
activation cache at Drive (`ACTS_CACHE_DIR`), and registers the NDIF key with nnsight (persisted to
nnsight's config so the subprocess runs in the next cell are authenticated).

In [ ]:
import os
from google.colab import userdata

os.environ['NDIF_API_KEY']   = userdata.get('NDIF_API_KEY')
os.environ['HF_TOKEN']       = userdata.get('HF_TOKEN')
os.environ['ACTS_CACHE_DIR'] = ACTS_CACHE_DIR  # read by Activations.from_model (get_acts_cache_dir)

import nnsight
nnsight.CONFIG.set_default_api_key(os.environ['NDIF_API_KEY'])
print('NDIF key + HF token configured; cache ->', os.environ['ACTS_CACHE_DIR'])

## 6. Run extraction (loops over all models)

Runs one experiment per model, writing to Drive. **tqdm progress bars stream into this cell's output**,
and you can watch cached batch files / `*_scores.json` appear under the Drive folders.

**Resume:** if Colab disconnects, just reconnect and re-run this cell — finished models are skipped
(their `detector.pt` / scores already exist) and, within a model, completed activation batches load
from `ACTS_CACHE_DIR` instead of being re-downloaded from NDIF.

In [ ]:
import subprocess

for model in MODELS:
    print(f"\n{'='*72}\n=== Extracting activations for: {model}\n{'='*72}", flush=True)
    cmd = [
        'python', 'deception_detection/scripts/experiment.py', 'run',
        '--config-file', CONFIG_FILE,
        f'--model_name={model}',
        f'--folder={OUT}',
    ]
    subprocess.run(cmd, cwd=WORKDIR, check=True)

print('\nAll models done. Outputs are on Drive at', OUT)

## Notes

- **Watch progress**: tqdm bars above + browse the Drive folders (`results/`, `activations_cache/`).
- **Change code**: edit in VS Code → `git push` → re-run cell **3** (`git pull`) → re-run cell **6**.
- **Resume granularity**: the cache unit is a *batch* (`BATCH_SIZE_ACTS`, default 32); a mid-batch
  interruption re-downloads only that batch (atomic writes prevent corrupt partial caches).
- **Colab idle**: sessions disconnect on idle (~90 min) or at runtime caps — this is expected; the
  resume design above makes it harmless. Don't use anti-idle scripts (against Colab ToS).